#Initilization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

#Reading the Data

In [0]:
df = spark.table("databricks_lakehouse.bronze.crm_sales_details")
df.display()

#Transformation

##Trimming the strings

In [0]:
trim_plan = {
    field.name: F.trim(F.col(field.name))
    for field in df.schema.fields
    if isinstance(field.dataType, StringType)
}

df = df.withColumns(trim_plan)
df.display()

## Fixing the date columns datatype

In [0]:
'''
df.filter( 
    (F.length(F.col("sls_order_dt").cast("string")) < 8) |
    (F.length(F.col("sls_due_dt").cast("string")) < 8) |
    (F.length(F.col("sls_ship_dt").cast("string")) < 8)
).display()
'''

df = (
    df.withColumn("sls_order_dt", 
        F.when(F.length(F.col("sls_order_dt").cast("string"))!=8, None)
        .otherwise(F.to_date(F.col("sls_order_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn("sls_ship_dt", 
        F.when(F.length(F.col("sls_ship_dt").cast("string"))!=8, None)
        .otherwise(F.to_date(F.col("sls_ship_dt").cast("string"), "yyyyMMdd"))
    )
    .withColumn("sls_due_dt", 
        F.when(F.length(F.col("sls_due_dt").cast("string"))!=8, None)
        .otherwise(F.to_date(F.col("sls_due_dt").cast("string"), "yyyyMMdd"))
    )
)
df.display()

##checking for null values


In [0]:
df.select(
    [F.count(F.when(F.col(c).isNull(),c)).alias(c)
    for c in df.columns]
).display()

##handling nulls in sales and price

In [0]:
df = (
        df.withColumn(
            "sls_sales", F.when(
                (F.col("sls_sales").isNull()) &
                (F.col("sls_price").isNotNull()) &
                (F.col("sls_quantity") != 0) , 
                F.col("sls_price")/ F.col("sls_quantity")
            ).otherwise(F.col("sls_sales"))
        )
        .withColumn(
            "sls_price", F.when(
                (F.col("sls_price").isNull()) &
                (F.col("sls_sales").isNotNull()) &
                (F.col("sls_quantity") != 0) ,
                F.col("sls_sales")* F.col("sls_quantity")
            ).otherwise(F.col("sls_price"))
        )

)
df.display()

##Renaming the columns 

In [0]:
rename_map = {
    field.name: field.name.replace("sls_", "").replace("prd", "product").replace("ord_", "order_").replace("cust_", "customer_").replace("dt", "date").replace("num", "number")
    for field in df.schema.fields
}

df = df.withColumnsRenamed(rename_map)
df.display()

#Writing the data

In [0]:
(
    df.write.mode("overwrite")
    .format("delta")
    .saveAsTable("databricks_lakehouse.silver.crm_sales")
)

#Reading the silver table

In [0]:
%sql
select * from databricks_lakehouse.silver.crm_sales limit 5